CELL 1 — Environment Fix

In [ ]:
!pip uninstall -y numpy ultralytics
!pip install "numpy==1.23.5" ultralytics --no-warn-conflicts

import os, torch, yaml, warnings, numpy as np
warnings.filterwarnings('ignore')

os.environ['WANDB_DISABLED'] = 'true'

print(f"NumPy Version: {np.__version__}")
print(f"Torch Version: {torch.__version__}")
print(f"GPUs Detected: {torch.cuda.device_count()}")

torch.manual_seed(42)


CELL 2 — Create Dataset YAML

In [ ]:
DATA_ROOT = '/kaggle/input/sku110dataset-resized640/v1.0/v1.0'

data_config = {
    'path': DATA_ROOT,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 1,
    'names': ['sku_product']
}

yaml_path = 'sku110_yolo.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print(open(yaml_path).read())


CELL 3 — Dataset Statistics

In [ ]:
from pathlib import Path
import cv2
from collections import Counter

def dataset_stats():
    print("=== DATASET INFO ===")
    for split in ['train','val','test']:
        imgs = list((Path(DATA_ROOT)/'images'/split).glob("*.jpg"))
        print(f"{split}: {len(imgs)} images")

dataset_stats()


CELL 4 — Verify Image Resolution (640×640)

In [ ]:
from tqdm.auto import tqdm

def check_integrity(split):
    img_dir = Path(DATA_ROOT)/'images'/split
    lbl_dir = Path(DATA_ROOT)/'labels'/split
    missing = []

    for p in tqdm(list(img_dir.glob("*.jpg")), desc=f"Checking {split}"):
        if not (lbl_dir/(p.stem + ".txt")).exists():
            missing.append(p.name)

    print(f"{split}: missing labels = {len(missing)}")

for s in ['train','val','test']:
    check_integrity(s)


CELL 5 — Dataset Integrity Check

In [ ]:
from tqdm.auto import tqdm

def check_integrity(split):
    img_dir = Path(DATA_ROOT)/'images'/split
    lbl_dir = Path(DATA_ROOT)/'labels'/split
    missing = []

    for p in tqdm(list(img_dir.glob("*.jpg")), desc=f"Checking {split}"):
        if not (lbl_dir/(p.stem + ".txt")).exists():
            missing.append(p.name)

    print(f"{split}: missing labels = {len(missing)}")

for s in ['train','val','test']:
    check_integrity(s)


CELL 6 — Bounding Box Statistics

In [ ]:
import numpy as np

def bbox_stats(split):
    ws, hs = [], []
    for p in (Path(DATA_ROOT)/'labels'/split).glob("*.txt"):
        for ln in open(p):
            _,_,_,w,h = map(float, ln.split())
            ws.append(w)
            hs.append(h)

    print(f"{split.upper()} — mean w={np.mean(ws):.4f}, mean h={np.mean(hs):.4f}")

bbox_stats("train")
bbox_stats("val")


CELL 7 — Load YOLO Model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")
print("YOLOv8-M loaded successfully")


CELL 8 — Hyperparameter Experiments (3 Params Each)

In [ ]:
hparam_experiments = [
    {
        "name": "YOLO_LR1e-3_B8_ADAMW",
        "lr0": 1e-3,
        "batch": 8,
        "optimizer": "AdamW"
    },
    {
        "name": "YOLO_LR5e-4_B8_ADAMW",
        "lr0": 5e-4,
        "batch": 8,
        "optimizer": "AdamW"
    },
    {
        "name": "YOLO_LR1e-3_B16_SGD",
        "lr0": 1e-3,
        "batch": 16,
        "optimizer": "SGD"
    }
]

hparam_experiments


CELL 9 — Run Hyperparameter Experiments (10 Epochs)

In [ ]:
from ultralytics import YOLO

def run_hparam_exp(cfg):

    fresh_model = YOLO("yolov8m.pt")

    print("\n=======================================")
    print(f" RUNNING EXPERIMENT: {cfg['name']}")
    print("---------------------------------------")
    print(f" lr0       = {cfg['lr0']}")
    print(f" batch     = {cfg['batch']}")
    print(f" optimizer = {cfg['optimizer']}")
    print(f" epochs    = 10")
    print("=======================================\n")

    return fresh_model.train(
        data=yaml_path,
        epochs=10,
        imgsz=640,
        batch=cfg["batch"],
        lr0=cfg["lr0"],
        optimizer=cfg["optimizer"],
        project="HParam_Experiments_YOLO",
        name=cfg["name"],
        device=[0,1],
        val=True,
        exist_ok=True,
        verbose=True
    )

for exp in hparam_experiments:
    run_hparam_exp(exp)

print("All YOLO hyperparameter experiments completed.")


CELL 10 — Select Best Experiment (mAP50-95)

In [ ]:
import pandas as pd

def get_map(exp_name):
    path = f"/kaggle/working/HParam_Experiments_YOLO/{exp_name}/results.csv"
    df = pd.read_csv(path)
    col = [c for c in df.columns if "map50-95" in c.lower()][0]
    return df[col].max()

scores = {}
for exp in hparam_experiments:
    scores[exp["name"]] = get_map(exp["name"])
    print(f"{exp['name']} → mAP50-95 = {scores[exp['name']]:.4f}")

best_exp = max(scores, key=scores.get)

print("\n==============================")
print(" BEST YOLO EXPERIMENT")
print("==============================")
print(best_exp)


CELL 11 — Visualization (All Experiments)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_full_training_results(csv_path, title="Training Visualization"):
    df = pd.read_csv(csv_path)
    epochs = df["epoch"]

    # Dynamically detect available loss columns
    train_loss_cols = [c for c in df.columns if "train" in c and "loss" in c]
    val_loss_cols = [c for c in df.columns if "val" in c and "loss" in c]

    # Dynamically detect precision, recall, mAP
    prec_cols = [c for c in df.columns if "precision" in c.lower()]
    recall_cols = [c for c in df.columns if "recall" in c.lower()]
    map50_cols = [c for c in df.columns if "map50" in c.lower()]
    map5095_cols = [c for c in df.columns if "map50-95" in c.lower() or "mAP50-95" in c]

    # detect LR
    lr_cols = [c for c in df.columns if "lr" in c.lower()]

    plt.figure(figsize=(20, 16))

    # -------------------------
    # 1) Training vs Validation LOSS
    # -------------------------
    plt.subplot(3, 2, 1)
    for col in train_loss_cols:
        plt.plot(epochs, df[col], label=f"Train {col.split('/')[-1]}")
    for col in val_loss_cols:
        plt.plot(epochs, df[col], label=f"Val {col.split('/')[-1]}")
    plt.title(f"{title} — Loss Curves")
    plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 2) Precision
    # -------------------------
    if prec_cols:
        plt.subplot(3, 2, 2)
        plt.plot(epochs, df[prec_cols[0]], label="Precision")
        plt.title(f"{title} — Precision")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 3) Recall
    # -------------------------
    if recall_cols:
        plt.subplot(3, 2, 3)
        plt.plot(epochs, df[recall_cols[0]], label="Recall")
        plt.title(f"{title} — Recall")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 4) mAP50
    # -------------------------
    if map50_cols:
        plt.subplot(3, 2, 4)
        plt.plot(epochs, df[map50_cols[0]], label="mAP50")
        plt.title(f"{title} — mAP50")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 5) mAP50-95
    # -------------------------
    if map5095_cols:
        plt.subplot(3, 2, 5)
        plt.plot(epochs, df[map5095_cols[0]], label="mAP50-95")
        plt.title(f"{title} — mAP50-95")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 6) Learning Rate
    # -------------------------
    if lr_cols:
        plt.subplot(3, 2, 6)
        plt.plot(epochs, df[lr_cols[0]], label="Learning Rate")
        plt.title(f"{title} — LR Curve")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
print("=== VISUALIZATION FOR ALL EXPERIMENTS ===")

for exp in hparam_experiments:
    csv_path = f"/kaggle/working/HParam_Experiments_YOLO/{exp['name']}/results.csv"
    print(f"\n### Visualizing {exp['name']} ###")
    plot_full_training_results(csv_path, exp['name'])


CELL 12 — Retrain BEST Experiment (25 Epochs)

In [ ]:
best_cfg = [e for e in hparam_experiments if e["name"] == best_exp][0]

final_model = YOLO("yolov8m.pt")

final_model.train(
    data=yaml_path,
    epochs=25,
    imgsz=640,
    batch=best_cfg["batch"],
    lr0=best_cfg["lr0"],
    optimizer=best_cfg["optimizer"],
    project="Final_Main_Model_YOLO",
    name="YOLO_Final",
    device=[0,1],
    val=True,
    exist_ok=True
)


In [ ]:
# ============================
# CELL 14 — Full Training Visualization (Loss + mAP + LR)
# ============================

import pandas as pd
import matplotlib.pyplot as plt

def plot_full_training_results(csv_path, title="Training Visualization"):
    df = pd.read_csv(csv_path)
    epochs = df["epoch"]

    # Dynamically detect available loss columns
    train_loss_cols = [c for c in df.columns if "train" in c and "loss" in c]
    val_loss_cols = [c for c in df.columns if "val" in c and "loss" in c]

    # Dynamically detect precision, recall, mAP
    prec_cols = [c for c in df.columns if "precision" in c.lower()]
    recall_cols = [c for c in df.columns if "recall" in c.lower()]
    map50_cols = [c for c in df.columns if "map50" in c.lower()]
    map5095_cols = [c for c in df.columns if "map50-95" in c.lower() or "mAP50-95" in c]

    # detect LR
    lr_cols = [c for c in df.columns if "lr" in c.lower()]

    plt.figure(figsize=(20, 16))

    # -------------------------
    # 1) Training vs Validation LOSS
    # -------------------------
    plt.subplot(3, 2, 1)
    for col in train_loss_cols:
        plt.plot(epochs, df[col], label=f"Train {col.split('/')[-1]}")
    for col in val_loss_cols:
        plt.plot(epochs, df[col], label=f"Val {col.split('/')[-1]}")
    plt.title(f"{title} — Loss Curves")
    plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 2) Precision
    # -------------------------
    if prec_cols:
        plt.subplot(3, 2, 2)
        plt.plot(epochs, df[prec_cols[0]], label="Precision")
        plt.title(f"{title} — Precision")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 3) Recall
    # -------------------------
    if recall_cols:
        plt.subplot(3, 2, 3)
        plt.plot(epochs, df[recall_cols[0]], label="Recall")
        plt.title(f"{title} — Recall")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 4) mAP50
    # -------------------------
    if map50_cols:
        plt.subplot(3, 2, 4)
        plt.plot(epochs, df[map50_cols[0]], label="mAP50")
        plt.title(f"{title} — mAP50")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 5) mAP50-95
    # -------------------------
    if map5095_cols:
        plt.subplot(3, 2, 5)
        plt.plot(epochs, df[map5095_cols[0]], label="mAP50-95")
        plt.title(f"{title} — mAP50-95")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    # -------------------------
    # 6) Learning Rate
    # -------------------------
    if lr_cols:
        plt.subplot(3, 2, 6)
        plt.plot(epochs, df[lr_cols[0]], label="Learning Rate")
        plt.title(f"{title} — LR Curve")
        plt.xlabel("Epoch"); plt.grid(); plt.legend()

    plt.tight_layout()
    plt.show()

# Run Visualization for Final Model
final_csv = "/kaggle/working/Final_Main_Model_YOLO/YOLO_Final/results.csv"
plot_full_training_results(final_csv, "YOLO Final MODEL")


CELL 13 — Load Final Model

In [ ]:
final_model = YOLO("/kaggle/working/Final_Main_Model_YOLO/YOLO_Final/weights/best.pt")
print("Final YOLO model loaded")


CELL 14 — Final Test Evaluation

In [ ]:
# ============================
# CELL 15 — Final Test Evaluation (Metrics Only)
# ============================

final_metrics = final_model.val(
    data=yaml_path,
    split='test',
    device=[0,1]
)

print("\n=======================================")
print(" FINAL TEST METRICS")
print("=======================================\n")
print(f"mAP50       : {final_metrics.box.map50:.4f}")
print(f"mAP50-95    : {final_metrics.box.map:.4f}")
print(f"Precision   : {final_metrics.box.mp:.4f}")
print(f"Recall      : {final_metrics.box.mr:.4f}")

# F1 score computation
f1 = 2 * (final_metrics.box.mp * final_metrics.box.mr) / (final_metrics.box.mp + final_metrics.box.mr)
print(f"F1 Score    : {f1:.4f}")


In [ ]:
# ============================
# CELL 16 — Generate Evaluation Plots (PR, F1, Confidence curves)
# ============================

eval_results = final_model.val(
    data=yaml_path,
    split='test',
    save_json=True,
    save_hybrid=True,
    plots=True,   # <--- Generates PR, F1, Confidence curves
    device=[0,1]
)

print("\nEvaluation plots saved in:")
print(eval_results.save_dir)


In [ ]:
# ============================
# CELL 17 — Speed Test (Inference Latency)
# ============================

from pathlib import Path
import random

test_images = list((Path(DATA_ROOT)/"images"/"test").glob("*.jpg"))
test_img = random.choice(test_images)

speed_results = final_model.predict(
    source=str(test_img),
    device=0,
    conf=0.25,
    verbose=False
)

print("\n=======================================")
print(" MODEL SPEED TEST (ms)")
print("=======================================\n")
print(speed_results[0].speed)


CELL 15 — Sample Predictions

In [ ]:
# ============================
# CELL 18 — Sample Predictions Visualization
# ============================

import matplotlib.pyplot as plt
import cv2
import numpy as np

sample_imgs = random.sample(test_images, 4)

plt.figure(figsize=(18,14))

for i, img_path in enumerate(sample_imgs, start=1):
    pred = final_model.predict(source=str(img_path), conf=0.25, device=0)[0]
    img = pred.plot()   # BGR

    plt.subplot(2,2,i)
    plt.imshow(img[..., ::-1])   # Convert BGR → RGB
    plt.title(f"Prediction: {img_path.name}")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ============================
# CELL 19 — Summary Table (Copy into Report)
# ============================

print("\n=======================================")
print(" SUMMARY TABLE — FINAL MODEL")
print("=======================================\n")

print(f"""
MODEL NAME                 : RT-DETR (Final Best Model)
TEST IMAGES                : {len(test_images)}
mAP50                      : {final_metrics.box.map50:.4f}
mAP50-95                   : {final_metrics.box.map:.4f}
Precision                  : {final_metrics.box.mp:.4f}
Recall                     : {final_metrics.box.mr:.4f}
F1 Score                   : {f1:.4f}

Inference Speed (Preprocess): {speed_results[0].speed['preprocess']:.2f} ms
Inference Speed (Model)     : {speed_results[0].speed['inference']:.2f} ms
""")


In [ ]:
# ============================
# CELL 17 — Learning Curve (Final Model)
# ============================

df = pd.read_csv("/kaggle/working/Final_Main_Model_YOLO/YOLO_Final/results.csv")
map_col = [c for c in df.columns if "mAP" in c][0]

plt.plot(df['epoch'], df[map_col])
plt.title("Learning Curve — Main Model")
plt.xlabel("Epoch")
plt.ylabel(map_col)
plt.grid()
plt.show()


In [ ]:
# ============================
# CELL 16 — Overfitting Check
# ============================

df = pd.read_csv("/kaggle/working/Final_Main_Model_YOLO/YOLO_Final/results.csv")
train_loss = df['train/box_loss'].iloc[-1]
val_loss = df['val/box_loss'].iloc[-1]

print("Train Loss:", train_loss)
print("Val Loss:", val_loss)
print("Gap:", val_loss - train_loss)


In [ ]:
!zip -r val9.zip /kaggle/working/runs/detect/val9


In [ ]:
from IPython.display import FileLink
FileLink('val9.zip')
